## Factors associated with delayed cerebral ischemia (DCI) timing
- variables associated with time to DCI
- predictors of occurrence of late DCI?

In [ ]:
import numpy as np
import pandas as pd
import datetime
import os

from utils.utils import load_encrypted_xlsx, safe_conversion_to_datetime

In [ ]:
foch_registry_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/foch_data/HSA_dci_manual_extraction.xlsx'
post_hoc_corrected_registry_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/sos_sah_data/post_hoc_modified_aSAH_DATA_2009_2023_24122023.xlsx'

In [ ]:
kssg_registry_df = load_encrypted_xlsx(post_hoc_corrected_registry_path)
foch_registry_df = pd.read_excel(foch_registry_data_path)
print(f'Excluding {foch_registry_df["exclude_yn"].sum()} patients because not aSAH')
foch_registry_df = foch_registry_df[foch_registry_df['exclude_yn'] == 0]

## Preprocessing

In [ ]:
# for patients with Date_CVS_Start nan but Date_CVS_DSA not nan, set Date_CVS_Start = Date_CVS_DSA
kssg_registry_df.loc[(kssg_registry_df['Date_CVS_Start'].isnull()) & (
    kssg_registry_df['Date_CVS_DSA'].notnull()), 'Date_CVS_Start'] = kssg_registry_df['Date_CVS_DSA']
# for patients with Date_CVS_Start nan but Date_CVS_CTA not nan, set Date_CVS_Start = Date_CVS_CTA
kssg_registry_df.loc[(kssg_registry_df['Date_CVS_Start'].isnull()) & (
    kssg_registry_df['Date_CVS_CTA'].notnull()), 'Date_CVS_Start'] = kssg_registry_df['Date_CVS_CTA']
# for patients with Date_CVS_Start nan but Date_CVS_TCD not nan, set Date_CVS_Start = Date_CVS_TCD
kssg_registry_df.loc[(kssg_registry_df['Date_CVS_Start'].isnull()) & (
    kssg_registry_df['Date_CVS_TCD'].notnull()), 'Date_CVS_Start'] = kssg_registry_df['Date_CVS_TCD']

# patients with Date_CVS_Start not na but with  but CVS_YN = 0, in this case we should set CVS_YN = 1
kssg_registry_df.loc[(kssg_registry_df['CVS_YN'] == 0) & (
    kssg_registry_df['Date_CVS_Start'].apply(safe_conversion_to_datetime).notnull()), 'CVS_YN'] = 1
# if Date_Ictus is nan, set it to Date_admission
kssg_registry_df.loc[kssg_registry_df['Date_Ictus'].isnull(), 'Date_Ictus'] = kssg_registry_df['Date_admission']

In [ ]:
n_dci_ischemia = kssg_registry_df['DCI_ischemia'].sum()
n_dci_infarct = kssg_registry_df['DCI_infarct'].sum()
n_cvs = kssg_registry_df['CVS_YN'].sum()

print('Number of patients with DCI ischemia: {}'.format(n_dci_ischemia))
print('Number of patients with DCI infarct: {}'.format(n_dci_infarct))
print('Number of patients with CVS: {}'.format(n_cvs))
foch_n_patients = foch_registry_df['PatientID'].nunique()
print('Number of patients in Foch registry: {}'.format(foch_n_patients))
foch_n_dci = foch_registry_df['DCI_YN'].sum()
print('Number of patients with DCI in Foch registry: {}'.format(foch_n_dci))

In [ ]:
 # convert to datetime
foch_registry_df['DCI_exact_date'] = foch_registry_df.DCI_exact_date.str.replace(':', 'h')
foch_registry_df['DCI_exact_date'] = pd.to_datetime(foch_registry_df['DCI_exact_date'], format='%d/%m/%Y à %Hh%M')
foch_registry_df["AdmissionTime"] = pd.to_datetime(foch_registry_df["AdmissionTime"])
foch_registry_df['Date_ictus'] = pd.to_datetime(foch_registry_df['Date_ictus'])

assert foch_registry_df['DCI_YN'].dropna().apply(lambda x: x in [0, 1]).all()
foch_registry_df['DCI_YN'] = foch_registry_df['DCI_YN'].astype(int, errors='ignore')

In [ ]:
def registry_date_conversion(date):
    if isinstance(date, datetime.datetime):
        return date
    elif type(date) == str:
        return datetime.datetime.strptime(date, '%d.%m.%Y').date()
    else:
        return date

In [ ]:
# add Date_DCI_ischemia_first_image and Time_DCI_ischemia_first_image to get the full date
kssg_registry_df['full_date_dci_ischemia'] = kssg_registry_df['Date_DCI_ischemia_first_image'].apply(
    registry_date_conversion).astype(str) + ' ' + kssg_registry_df['Time_DCI_ischemia_first_image'].astype(str)
# replace NaT nan with nan
kssg_registry_df['full_date_dci_ischemia'] = kssg_registry_df['full_date_dci_ischemia'].replace('NaT nan',
                                                                                                        pd.NaT)
kssg_registry_df['full_date_dci_ischemia'] = kssg_registry_df['full_date_dci_ischemia'].apply(
    safe_conversion_to_datetime)

kssg_registry_df['full_date_dci_infarction'] = kssg_registry_df['Date_DCI_infarct_first_image'].astype(
    str) + ' ' + kssg_registry_df['Time_DCI_infarct_first_image'].astype(str)
# replace NaT nan with nan
kssg_registry_df['full_date_dci_infarction'] = kssg_registry_df['full_date_dci_infarction'].replace('NaT nan',
                                                                                                            pd.NaT)
kssg_registry_df['full_date_dci_infarction'] = kssg_registry_df['full_date_dci_infarction'].apply(
    safe_conversion_to_datetime)

# ensure number of nans in full_date_dci_ischemia and Date_DCI_ischemia_first_image are the same
assert kssg_registry_df['full_date_dci_ischemia'].isnull().sum() == kssg_registry_df[
    'Date_DCI_ischemia_first_image'].isnull().sum()
# ensure number of nans in full_date_dci_infarction and Date_DCI_infarct_first_image are the same
assert kssg_registry_df['full_date_dci_infarction'].isnull().sum() == kssg_registry_df[
    'Date_DCI_infarct_first_image'].isnull().sum()
# compute time to CVS, DCI ischemia and DCI infarction
kssg_registry_df['time_to_cvs'] = kssg_registry_df['Date_CVS_Start'].apply(safe_conversion_to_datetime) - \
                                      kssg_registry_df['Date_Ictus'].apply(safe_conversion_to_datetime)

kssg_registry_df['time_to_dci_ischemia'] = kssg_registry_df['full_date_dci_ischemia'] - kssg_registry_df[
    'Date_Ictus'].apply(safe_conversion_to_datetime)
kssg_registry_df['time_to_dci_infarction'] = kssg_registry_df['full_date_dci_infarction'] - \
                                                 kssg_registry_df['Date_Ictus'].apply(safe_conversion_to_datetime)

In [ ]:
# check if any negative timings
print('Number of negative time_to_cvs: {}'.format((kssg_registry_df['time_to_cvs'] < pd.Timedelta(0)).sum()))
print('Number of negative time_to_dci_ischemia: {}'.format((kssg_registry_df['time_to_dci_ischemia'] < pd.Timedelta(0)).sum()))
print('Number of negative time_to_dci_infarction: {}'.format((kssg_registry_df['time_to_dci_infarction'] < pd.Timedelta(0)).sum()))

# filter out negative times
kssg_registry_df.loc[kssg_registry_df['time_to_cvs'] < pd.Timedelta(0), 'time_to_cvs'] = pd.NaT
kssg_registry_df.loc[kssg_registry_df['time_to_dci_ischemia'] < pd.Timedelta(0), 'time_to_dci_ischemia'] = pd.NaT
kssg_registry_df.loc[kssg_registry_df['time_to_dci_infarction'] < pd.Timedelta(0), 'time_to_dci_infarction'] = pd.NaT


In [ ]:
foch_registry_df['time_to_dci_ischemia'] = foch_registry_df['DCI_exact_date'] - foch_registry_df['Date_ictus']
print('Number of negative time_to_dci_ischemia in Foch: {}'.format(
    (foch_registry_df['time_to_dci_ischemia'] < pd.Timedelta(0)).sum()))

In [ ]:
kssg_registry_df['time_to_cvs_days'] = kssg_registry_df['time_to_cvs'].dt.total_seconds() / (60*60*24)
kssg_registry_df['time_to_dci_ischemia_days'] = kssg_registry_df['time_to_dci_ischemia'].dt.total_seconds() / (60*60*24)
kssg_registry_df['time_to_dci_infarction_days'] = kssg_registry_df['time_to_dci_infarction'].dt.total_seconds() / (60*60*24)
foch_registry_df['time_to_dci_ischemia_days'] = foch_registry_df['time_to_dci_ischemia'].dt.total_seconds() / (60*60*24)

Preproccess time to death

In [ ]:
foch_registry_df['time_to_death'] = pd.to_datetime(foch_registry_df.death_time) - foch_registry_df.Date_ictus
foch_registry_df['time_to_death_days'] = foch_registry_df['time_to_death'].dt.total_seconds() / (60*60*24)

# if Death 1 and Date_Death is NaT, set Date_Death to Date_Discharge
kssg_registry_df.loc[(kssg_registry_df['Death'] == 1) & (kssg_registry_df['Date_Death'].isna()), 'Date_Death'] = kssg_registry_df.loc[(kssg_registry_df['Death'] == 1) & (kssg_registry_df['Date_Death'].isna()), 'Date_Discharge']

kssg_registry_df['time_to_death'] = pd.to_datetime(kssg_registry_df['Date_Death']) - kssg_registry_df['Date_Ictus'].apply(safe_conversion_to_datetime)
kssg_registry_df['time_to_death_days'] = kssg_registry_df['time_to_death'].dt.total_seconds() / (60*60*24)


## Joint analysis of KSSG and Foch data

In [ ]:
foch_registry_df['Coiling'] = (foch_registry_df['treatment'] == 'coil').astype(int)
foch_registry_df['Clipping'] = (foch_registry_df['treatment'] == 'clip').astype(int)

In [ ]:
risk_factors = ['initial_GCS', 'fischer', 'wfns', 'location', 'Coiling', 'Clipping']
time_to_event = ['time_to_dci_ischemia_days', 'time_to_death_days']

In [ ]:
temp_kssg_df = kssg_registry_df.rename(columns={'GCS_admission': 'initial_GCS', 'WFNS': 'wfns', 'Fisher_Score': 'fischer', 'Aneurysm_Artery_Code': 'location'})

In [ ]:
joint_df = pd.concat([temp_kssg_df[risk_factors + time_to_event], foch_registry_df[risk_factors + time_to_event]])

In [ ]:
joint_df.fischer = pd.to_numeric(joint_df.fischer, errors='coerce')

In [ ]:
# encode location
# - acoma = 0
# - acm = 1
# - acomp / pca = 2
# - aci = 3
# - aca = 4
# - tb and branches = 5

# mapping = {
#         [8, 'acoma']: 0,
#         ['acm', 7, 20, 21]: 1,
#         ['acomp', 27]: 2,
#         ['aci', 1,2,3,4,5,6,25, 18, 19, 25, 26, 29, 31]: 3,
#         ['aca', 9, 22, 24]: 4,
#         ['tb', 'PICA', 'pica', 10, 11, 12, 13, 14, 15, 16, 17]: 5,
#         ['pca', 23, 28]: 2
#      }
# for all location entries, split at , and take the first element, then strip all spaces
joint_df['location'] = joint_df['location'].apply(lambda x: x.split(',')[0].split('(')[0].strip() if type(x) == str else x)

joint_df['location_encoded'] = joint_df['location'].map(
    {
        8: 0, 'acoma': 0,
        'acm': 1, 7: 1, 20: 1, 21: 1,
        'acomp': 2, 27: 2,
        'aci': 3, 1: 3, 2: 3, 3: 3, 4: 3, 5: 3, 6: 3, 25: 3, 18: 3, 19: 3, 25: 3, 26: 3, 29: 3, 31: 3,
        'aca': 4, 9: 4, 22: 4, 24: 4,
        'tb': 5, 'PICA': 5, 'pica': 5, 10: 5, 11: 5, 12: 5, 13: 5, 14: 5, 15: 5, 16: 5, 17: 5,
        'pca': 2, 23: 2, 28: 2,

    #     as well as all string entries
        '8': 0,
        '7': 1, '20': 1, '21': 1,
        '27': 2,
        '1': 3, '2': 3, '3': 3, '4': 3, '5': 3, '6': 3, '25': 3, '18': 3, '19': 3, '25': 3, '26': 3, '29': 3, '31': 3,
        '9': 4, '22': 4, '24': 4,
        '10': 5, '11': 5, '12': 5, '13': 5, '14': 5, '15': 5, '16': 5, '17': 5,
        '23': 2, '28': 2
    })

In [ ]:
joint_df.location_encoded.value_counts()


In [ ]:
# encode event: wichever occors first 1 for DCI / 2 for death / nan if none
# add time to event column

def define_event(row):
    if pd.isna(row['time_to_dci_ischemia_days']) and pd.isna(row['time_to_death_days']):
        return np.nan
    # if only one is nan, return the other
    elif pd.isna(row['time_to_dci_ischemia_days']):
        return 2
    elif pd.isna(row['time_to_death_days']):
        return 1
    # if both are not nan, return the one that occurs first
    else:
        return 1 if row['time_to_dci_ischemia_days'] < row['time_to_death_days'] else 2


joint_df['event'] = joint_df.apply(define_event, axis=1)
joint_df['time_to_event'] = joint_df.apply(lambda x: x['time_to_dci_ischemia_days'] if x['event'] == 1 else x['time_to_death_days'], axis=1)

In [ ]:
joint_df

In [ ]:
# print number of nans for all risk_factors with events non null
for rf in risk_factors:
    print(f'Number of nans for {rf}: {joint_df[joint_df.event.notnull() ][rf].isnull().sum()}')

# print number of columns with any nan in risk_factors
print('Number of rows with any nan in risk_factors: {}'.format(joint_df[joint_df.event.notnull()][risk_factors].isnull().any(axis=1).sum()))

## Competing risks analysis

In [ ]:
clean_df = joint_df.copy()
clean_df.drop(columns=['location', 'time_to_dci_ischemia_days', 'time_to_death_days'], inplace=True)
clean_df.dropna(inplace=True)

In [ ]:
rfs = ['initial_GCS', 'fischer', 'wfns', 'location_encoded', 'Coiling', 'Clipping']
joint_dci_df = joint_df[joint_df.event == 1].reset_index(drop=True)

In [ ]:
joint_dci_df

In [ ]:
# regression
import statsmodels.api as sm

temp_df = joint_dci_df.dropna(subset=rfs)
X = temp_df[rfs]
X = sm.add_constant(X)
y = temp_df['time_to_event']
model = sm.OLS(y, X.astype(float)).fit()
model.summary()


## Plotting

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# plot a boxplot for every risk factor (x-axis) against time_to_event (y-axis)
fig, axs = plt.subplots(2, 3, figsize=(20, 10))
for i, rf in enumerate(rfs):
    sns.boxplot(data=joint_dci_df, x=rf, y='time_to_event', hue=rf,
                ax=axs[i//3, i%3])
    axs[i//3, i%3].set_title(rf)
plt.show()